# Customer Cohort Retention + Price-Volume-Mix Bridge

Data: [Online Retail II](https://doi.org/10.24432/C5CG6D) (UCI), a UK online gift retailer, Dec 2009 to Dec 2011.

**Before you start:** run `python download_data.py`. Until you do, this notebook uses a small synthetic sample so everything runs. Don't report findings from the sample.

Cells marked **TODO** are yours to complete.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from retail import load_transactions, clean_transactions, add_cohorts, retention_table, pvm_bridge

REAL = Path("data/online_retail_ii.csv.gz")
SAMPLE = Path("data/sample_transactions.csv")
DATA_PATH = REAL if REAL.exists() else SAMPLE
print("Using:", DATA_PATH, "(SYNTHETIC SAMPLE)" if DATA_PATH == SAMPLE else "")

raw = load_transactions(DATA_PATH)
print(f"{len(raw):,} rows, {raw['invoice_date'].min():%Y-%m-%d} to {raw['invoice_date'].max():%Y-%m-%d}")
raw.head()

## 1. Cleaning

In [ ]:
df, cleaning_log = clean_transactions(raw)
cleaning_log

### TODO 1: Review the cleaning rules

Every rule in `clean_transactions` is a judgment call. Look at the log and the raw data, then answer:

- Removing cancellation rows leaves the original sale in the data. Does that overstate revenue? What would you do about it?
- Many customers are wholesalers. Should very large orders be treated differently?
- Is there anything the rules miss?

Change the rules in `retail.py` if you disagree, and write down why.

*Your answer:*


In [ ]:
# TODO 1: explore here


## 2. Cohort retention

In [ ]:
cohorts = add_cohorts(df)
retention = retention_table(cohorts)

first_12 = retention.loc[:, 0:12]
fig, ax = plt.subplots(figsize=(12, 0.35 * len(first_12) + 2))
im = ax.imshow(first_12.values, cmap="Blues", vmin=0, vmax=first_12.loc[:, 1:].max().max(), aspect="auto")
ax.set_xticks(range(first_12.shape[1]), first_12.columns)
ax.set_yticks(range(first_12.shape[0]), [str(p) for p in first_12.index])
ax.set_xlabel("Months since first purchase")
ax.set_ylabel("First purchase month (cohort)")
ax.set_title("Share of each cohort that bought again")
for i in range(first_12.shape[0]):
    for j in range(first_12.shape[1]):
        v = first_12.iat[i, j]
        if not np.isnan(v) and j > 0:
            ax.text(j, i, f"{v:.0%}", ha="center", va="center", fontsize=7)
plt.colorbar(im, ax=ax, format=lambda x, _: f"{x:.0%}")
plt.show()

### TODO 2: What does the retention table say?

- Compare month-1 and month-3 retention across cohorts. Are newer cohorts better or worse?
- Look closely at the **first** cohort (December 2009). Why might it look different from the others? Think about what "first purchase" means for customers who were already buying before the data starts.
- Why are the most recent cohorts' rows mostly blank?

*Your answer:*


In [ ]:
# TODO 2: your code here


### TODO 3: Revenue retention

Customer counts treat a £10 customer the same as a £10,000 wholesaler. Build the same table using **revenue** instead: each cohort's revenue in month N divided by its revenue in month 0. Does the story change?

In [ ]:
# TODO 3: your code here


## 3. Price-volume-mix bridge

Split the change in revenue between two periods into price, volume, mix, new products and lost products.

### TODO 4: Pick fair periods, then build the bridge

1. **Check the date range first.** Which two periods make a fair comparison? (Comparing full calendar years would be a mistake here. Why?)
2. Add a `period` column to `df` that labels each row's period (and drops rows outside both).
3. Implement `pvm_bridge` in `retail.py`. The docstring describes one common convention.
4. Check that it reconciles: base revenue + all components = current revenue.

The chart function below is ready to use once your bridge works.

In [ ]:
def plot_bridge(bridge: pd.Series, title: str = "Revenue bridge") -> None:
    """Waterfall chart for a Series ordered: base_revenue, components..., current_revenue."""
    labels = list(bridge.index)
    values = bridge.values.astype(float)
    bottoms, heights, colors = [], [], []
    running = 0.0
    for i, v in enumerate(values):
        if i == 0 or i == len(values) - 1:
            bottoms.append(0); heights.append(v); colors.append("tab:gray")
            running = v
        else:
            bottoms.append(running if v >= 0 else running + v)
            heights.append(abs(v)); colors.append("tab:green" if v >= 0 else "tab:red")
            running += v
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(labels, heights, bottom=bottoms, color=colors)
    for i, v in enumerate(values):
        ax.text(i, bottoms[i] + heights[i], f"{v:,.0f}", ha="center", va="bottom", fontsize=8)
    ax.set_ylabel("Revenue (£)")
    ax.set_title(title)
    plt.xticks(rotation=20)
    plt.show()

In [ ]:
# TODO 4: your code here
# df["period"] = ...
# bridge = pvm_bridge(df, period_col="period", base="...", current="...")
# assert abs(bridge.iloc[0] + bridge.iloc[1:-1].sum() - bridge.iloc[-1]) < 0.01, "Bridge doesn't reconcile"
# plot_bridge(bridge, "Revenue bridge: ... vs ...")


### TODO 5: Explain the bridge (your own words)

What drove the revenue change? Which component would you dig into next, and what question would you ask the business about it?

*Your answer:*


## Takeaways

<!-- TODO: 3 bullets with real numbers from the full dataset -->
